<h1 style="font-family: 'Times New Roman'; text-align: center; color: #1a1a2e;">
📋 PSL Data Analytics — Notebook 01: Data Cleaning (2016–2024)
</h1>

---

**Objective:** Produce a fully clean, consistently typed dataset for all 12 PSL records tables, ready for EDA and feature engineering.

**Output:** 12 cleaned CSV files saved to `../data/preprocessed/`


## 1. Import Libraries

In [45]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 20)

print("  Libraries imported successfully")
print(f"   pandas  {pd.__version__}")
print(f"   numpy   {np.__version__}")


  Libraries imported successfully
   pandas  3.0.3
   numpy   2.4.6


## 2. Load Raw Datasets

In [46]:
RAW  = '../data/raw/'
PROC = '../data/preprocessed/'

# ── Batting ──────────────────────────────────────────────────
most_runs          = pd.read_csv(RAW + 'most_runs_by_player.csv')
highest_score      = pd.read_csv(RAW + 'highest_individual_score.csv')
most_sixes_innings = pd.read_csv(RAW + 'most_sixes_in_innings.csv')
most_sixes_history = pd.read_csv(RAW + 'most_sixes_psl_history.csv')

# ── Bowling ──────────────────────────────────────────────────
most_wickets  = pd.read_csv(RAW + 'most_wickets_psl.csv')
best_bowling  = pd.read_csv(RAW + 'best_bowling_figures.csv')

# ── Fielding ─────────────────────────────────────────────────
most_catches    = pd.read_csv(RAW + 'most_catches_psl.csv')
most_dismissals = pd.read_csv(RAW + 'most_dismissals_wk.csv')

# ── Team ─────────────────────────────────────────────────────
highest_totals = pd.read_csv(RAW + 'highest_totals.csv')
lowest_totals  = pd.read_csv(RAW + 'lowest_totals.csv')
result_summary = pd.read_csv(RAW + 'result_summary_teams.csv')

# ── Timeline ─────────────────────────────────────────────────
match_wins = pd.read_csv(RAW + 'cumulative_match_wins.csv')

datasets = {
    'most_runs':          most_runs,
    'highest_score':      highest_score,
    'most_sixes_innings': most_sixes_innings,
    'most_sixes_history': most_sixes_history,
    'most_wickets':       most_wickets,
    'best_bowling':       best_bowling,
    'most_catches':       most_catches,
    'most_dismissals':    most_dismissals,
    'highest_totals':     highest_totals,
    'lowest_totals':      lowest_totals,
    'result_summary':     result_summary,
    'match_wins':         match_wins,
}

print("  All 12 datasets loaded successfully\n")
for name, df in datasets.items():
    print(f"  {name:<22}  shape: {str(df.shape):<12}")


  All 12 datasets loaded successfully

  most_runs               shape: (150, 14)   
  highest_score           shape: (100, 11)   
  most_sixes_innings      shape: (100, 11)   
  most_sixes_history      shape: (100, 15)   
  most_wickets            shape: (100, 15)   
  best_bowling            shape: (100, 11)   
  most_catches            shape: (100, 7)    
  most_dismissals         shape: (44, 9)     
  highest_totals          shape: (100, 9)    
  lowest_totals           shape: (200, 9)    
  result_summary          shape: (6, 15)     
  match_wins              shape: (278, 7)    


## 3. Dataset Overview

### 3.1 Shape, Nulls & Duplicates

In [47]:
summary_rows = []
for name, df in datasets.items():
    summary_rows.append({
        'Dataset'       : name,
        'Rows'          : df.shape[0],
        'Columns'       : df.shape[1],
        'Null Cells'    : int(df.isnull().sum().sum()),
        'Null %'        : round(df.isnull().sum().sum() / df.size * 100, 2),
        'Duplicate Rows': int(df.duplicated().sum()),
    })

overview = pd.DataFrame(summary_rows)
overview


,Dataset,Rows,Columns,Null Cells,Null %,Duplicate Rows
0,most_runs,150,14,0,0.0,0
1,highest_score,100,11,0,0.0,0
2,most_sixes_innings,100,11,0,0.0,0
3,most_sixes_history,100,15,0,0.0,0
4,most_wickets,100,15,0,0.0,0
5,best_bowling,100,11,0,0.0,0
6,most_catches,100,7,0,0.0,0
7,most_dismissals,44,9,0,0.0,0
8,highest_totals,100,9,0,0.0,0
9,lowest_totals,200,9,0,0.0,0


### 3.2 Column Names

In [48]:
for name, df in datasets.items():
    print(f"\n{'─'*60}")
    print(f"  {name}")
    print(f"{'─'*60}")
    print(df.columns.tolist())



────────────────────────────────────────────────────────────
  most_runs
────────────────────────────────────────────────────────────
['Player', 'Span', 'Mat', 'Inns', 'NO', 'Runs', 'HS', 'Ave', 'BF', 'SR', '100', '50', '0', '4s']

────────────────────────────────────────────────────────────
  highest_score
────────────────────────────────────────────────────────────
['Player', 'Runs', 'Mins', 'Balls', '4s', '6s', 'SR', 'Team', 'Opposition', 'Ground', 'Match Date']

────────────────────────────────────────────────────────────
  most_sixes_innings
────────────────────────────────────────────────────────────
['Player', 'Runs', 'Mins', 'Balls', '4s', '6s', 'SR', 'Team', 'Opposition', 'Ground', 'Match Date']

────────────────────────────────────────────────────────────
  most_sixes_history
────────────────────────────────────────────────────────────
['Player', 'Span', 'Mat', 'Inns', 'NO', 'Runs', 'HS', 'Ave', 'BF', 'SR', '100', '50', '0', '4s', '6s']

─────────────────────────────────────

### 3.3 Data Types & Non-Null Counts

In [49]:
for name, df in datasets.items():
    print(f"\n{'='*70}")
    print(f"  {name}")
    print(f"{'='*70}")
    df.info()



  most_runs
<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 14 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Player  150 non-null    str    
 1   Span    150 non-null    str    
 2   Mat     150 non-null    int64  
 3   Inns    150 non-null    int64  
 4   NO      150 non-null    str    
 5   Runs    150 non-null    int64  
 6   HS      150 non-null    str    
 7   Ave     150 non-null    float64
 8   BF      150 non-null    int64  
 9   SR      150 non-null    float64
 10  100     150 non-null    str    
 11  50      150 non-null    str    
 12  0       150 non-null    str    
 13  4s      150 non-null    int64  
dtypes: float64(2), int64(5), str(7)
memory usage: 21.6 KB

  highest_score
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Player      100 non-null    str    
 1   Runs

### 3.4 Statistical Summary

In [50]:
for name, df in datasets.items():
    print(f"\n{'='*70}")
    print(f"  {name}")
    print(f"{'='*70}")
    display(df.describe(include='all').T)



  most_runs


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Player,150,150,Babar Azam (IU/KK/PZ),1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Span,150,39,2016-2024,15,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Mat,150.0,NaN,NaN,NaN,29.5,23.944216,3.0,11.0,21.0,43.75,91.0
Inns,150.0,NaN,NaN,NaN,25.2,20.098082,3.0,10.0,18.0,36.0,88.0
NO,150,18,1,29,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Runs,150.0,NaN,NaN,NaN,537.093333,545.713772,94.0,174.0,325.0,687.75,3504.0
HS,150,103,73,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ave,150.0,NaN,NaN,NaN,25.393067,8.787963,8.47,19.93,25.155,29.5,54.5
BF,150.0,NaN,NaN,NaN,400.74,408.591531,66.0,141.5,249.5,497.25,2750.0
SR,150.0,NaN,NaN,NaN,132.727133,18.216254,85.95,121.41,132.83,144.5725,179.52



  highest_score


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Player,100,45,Babar Azam,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Runs,100,57,82,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Mins,100,32,-,54,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Balls,100.0,NaN,NaN,NaN,52.17,7.983639,27.0,46.75,54.0,58.0,67.0
4s,100.0,NaN,NaN,NaN,8.69,3.174058,3.0,6.0,8.0,11.0,20.0
6s,100,12,3,17,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SR,100.0,NaN,NaN,NaN,178.8067,32.04107,127.86,155.055,174.5,200.0,285.18
Team,100,6,United,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Opposition,100,6,v Zalmi,23,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ground,100,7,Karachi,29,NaN,NaN,NaN,NaN,NaN,NaN,NaN



  most_sixes_innings


...

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Team,200,6,Qalandars,40,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Score,200,146,121,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Overs,200.0,NaN,NaN,NaN,19.268,1.534454,10.2,19.3,20.0,20.0,20.0
RR,200.0,NaN,NaN,NaN,7.08065,0.806696,4.9,6.55,7.15,7.7775,9.05
Inns,200.0,NaN,NaN,NaN,1.415,0.493958,1.0,1.0,1.0,2.0,2.0
Opposition,200,6,v Zalmi,36,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ground,200,7,Dubai (DICS),59,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Result,200,3,lost,165,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Match Date,200,137,15 Jun 2021,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN



  result_summary


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Team,6,6,Islamabad United,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Span,6,2,2016-2024,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Mat,6.0,NaN,NaN,NaN,94.0,8.5557,79.0,92.5,94.5,98.75,104.0
Won,6.0,NaN,NaN,NaN,45.833333,7.782459,36.0,41.0,44.5,52.5,55.0
Lost,6.0,NaN,NaN,NaN,45.833333,8.20772,31.0,44.75,47.0,50.0,55.0
Draw,6.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Tied,6.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Tie+W,6.0,NaN,NaN,NaN,0.666667,0.516398,0.0,0.25,1.0,1.0,1.0
Tie+L,6.0,NaN,NaN,NaN,0.666667,0.816497,0.0,0.0,0.5,1.0,2.0
NR,6.0,NaN,NaN,NaN,1.0,0.894427,0.0,0.25,1.0,1.75,2.0



  match_wins


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Unnamed: 0,278,278,Match # 1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quetta,278.0,NaN,NaN,NaN,25.294964,11.681052,1.0,16.0,28.5,35.0,43.0
Karachi,278.0,NaN,NaN,NaN,20.406475,11.254372,0.0,11.0,22.0,30.0,37.0
Peshawar,278.0,NaN,NaN,NaN,29.841727,15.758676,0.0,15.0,31.0,44.0,56.0
Islamabad,278.0,NaN,NaN,NaN,29.02518,15.103831,0.0,17.0,29.0,41.0,56.0
Lahore,278.0,NaN,NaN,NaN,18.690647,13.208874,0.0,7.25,16.0,29.0,41.0
Multan,278.0,NaN,NaN,NaN,16.241007,14.442073,0.0,4.0,13.0,29.75,45.0


## 4. Issues Identified

After assessment the following issues were found:

| Dataset | Issues |
|---------|--------|
| `most_runs`, `most_sixes_history` | `HS` column contains `*` (not-out), `-` for century/50 counts; `Player` embeds team codes |
| `most_wickets` | `BBI` is a string ratio; `4` and `5` wicket haul cols use `-` for zero |
| `highest_score`, `most_sixes_innings`, `best_bowling`, `highest_totals`, `lowest_totals` | `Mins` and `Balls` contain `-` for missing; `Match Date` needs parsing |
| `highest_totals`, `lowest_totals` | `Score` column is `runs/wickets` string |
| `most_dismissals` | `Max Dis Inns` is a complex string |
| `result_summary` | `%` columns already float; `W/L` is a ratio string |
| `match_wins` | First column unnamed (match number); data is already numeric and clean |
| All career datasets | `Span` is a `YYYY-YYYY` string |


## 5. Data Cleaning

### 5.1 Helper Functions

In [51]:
import re

def clean_player_name(player_str):
    """Extract clean name from 'Babar Azam (IU/KK/PZ)' → 'Babar Azam'"""
    return re.sub(r'\s*\(.*?\)', '', str(player_str)).strip()

def extract_teams(player_str):
    """Extract team codes from 'Babar Azam (IU/KK/PZ)' → 'IU/KK/PZ'"""
    match = re.search(r'\(([^)]+)\)', str(player_str))
    return match.group(1) if match else None

def parse_span_start(span): 
    """'2016-2024' → 2016"""
    try: return int(str(span).split('-')[0])
    except: return np.nan

def parse_span_end(span):
    """'2016-2024' → 2024"""
    try: return int(str(span).split('-')[-1])
    except: return np.nan

def clean_hs(val):
    """'115*' → 115.0, '115' → 115.0"""
    return float(str(val).replace('*', '').strip()) if pd.notna(val) else np.nan

def is_not_out(val):
    """'115*' → True, '115' → False"""
    return '*' in str(val) if pd.notna(val) else False

def dash_to_zero(val):
    """'-' → 0, everything else stays"""
    if str(val).strip() == '-':
        return 0
    try: return float(val)
    except: return val

def parse_score_runs(score_str):
    """'262/3' → 262"""
    try: return int(str(score_str).split('/')[0])
    except: return np.nan

def parse_score_wickets(score_str):
    """'262/3' → 3, '199' → 10 (all out)"""
    parts = str(score_str).split('/')
    if len(parts) == 2:
        try: return int(parts[1])
        except: return np.nan
    return 10  # all out

def parse_date(date_str):
    """'08 Mar 2023' → pd.Timestamp"""
    try: return pd.to_datetime(str(date_str), dayfirst=True)
    except: return pd.NaT

print("Helper functions defined")


Helper functions defined


### 5.2 Clean: `most_runs`

In [52]:
df = most_runs.copy()

# Player name & team codes
df['player_name']  = df['Player'].apply(clean_player_name)
df['teams_played'] = df['Player'].apply(extract_teams)
df['num_teams']    = df['teams_played'].apply(lambda x: len(str(x).split('/')) if pd.notna(x) else 0)

# Span
df['span_start']   = df['Span'].apply(parse_span_start)
df['span_end']     = df['Span'].apply(parse_span_end)
df['career_yrs']   = df['span_end'] - df['span_start']

# HS — not-out flag + numeric
df['hs_not_out']   = df['HS'].apply(is_not_out)
df['HS']           = df['HS'].apply(clean_hs)

# Century/50/duck counts AND not-outs/fours: '-' → 0
for col in ['100', '50', '0', 'NO', '4s']:
    df[col] = df[col].apply(dash_to_zero).astype(float).astype('Int64')

# Drop original Player & Span cols (info preserved in new cols)
df = df.drop(columns=['Player', 'Span'])

# Rename for clarity
df = df.rename(columns={
    'Mat': 'matches', 'Inns': 'innings', 'NO': 'not_outs',
    'Runs': 'runs', 'HS': 'high_score', 'Ave': 'average',
    'BF': 'balls_faced', 'SR': 'strike_rate',
    '100': 'centuries', '50': 'fifties', '0': 'ducks', '4s': 'fours',
})

print("Shape:", df.shape)
print("\nCleaned dtypes:")
print(df.dtypes)
df.head(3)


Shape: (150, 19)

Cleaned dtypes:
matches           int64
innings           int64
not_outs          Int64
runs              int64
high_score      float64
average         float64
balls_faced       int64
strike_rate     float64
centuries         Int64
fifties           Int64
ducks             Int64
fours             Int64
player_name         str
teams_played        str
num_teams         int64
span_start        int64
span_end          int64
career_yrs        int64
hs_not_out         bool
dtype: object


,matches,innings,not_outs,runs,high_score,average,balls_faced,strike_rate,centuries,fifties,ducks,fours,player_name,teams_played,num_teams,span_start,span_end,career_yrs,hs_not_out
0,90,88,11,3504,115.0,45.50,2750,127.41,2,33,8,386,Babar Azam,IU/KK/PZ,3,2016,2024,8,False
1,84,84,1,2525,115.0,30.42,1800,140.27,2,19,5,237,Fakhar Zaman,LQ,1,2017,2024,7,False
2,83,72,14,2403,110.0,41.43,1885,127.48,1,20,5,216,Mohammad Rizwan,KK/LQ/MS,3,2016,2024,8,True


In [53]:
most_runs_clean = df.copy()
print(" most_runs cleaned")


 most_runs cleaned


### 5.3 Clean: `most_sixes_history`

In [54]:
df = most_sixes_history.copy()

df['player_name']  = df['Player'].apply(clean_player_name)
df['teams_played'] = df['Player'].apply(extract_teams)
df['span_start']   = df['Span'].apply(parse_span_start)
df['span_end']     = df['Span'].apply(parse_span_end)
df['career_yrs']   = df['span_end'] - df['span_start']
df['hs_not_out']   = df['HS'].apply(is_not_out)
df['HS']           = df['HS'].apply(clean_hs)

for col in ['100', '50', '0', 'NO']:
    df[col] = df[col].apply(dash_to_zero).astype(float).astype('Int64')

df = df.drop(columns=['Player', 'Span'])
df = df.rename(columns={
    'Mat': 'matches', 'Inns': 'innings', 'NO': 'not_outs',
    'Runs': 'runs', 'HS': 'high_score', 'Ave': 'average',
    'BF': 'balls_faced', 'SR': 'strike_rate',
    '100': 'centuries', '50': 'fifties', '0': 'ducks',
    '4s': 'fours', '6s': 'sixes',
})

most_sixes_history_clean = df.copy()
print("most_sixes_history cleaned | shape:", df.shape)
df.head(3)


most_sixes_history cleaned | shape: (100, 19)


,matches,innings,not_outs,runs,high_score,average,balls_faced,strike_rate,centuries,fifties,ducks,fours,sixes,player_name,teams_played,span_start,span_end,career_yrs,hs_not_out
0,84,84,1,2525,115.0,30.42,1800,140.27,2,19,5,237,104,Fakhar Zaman,LQ,2017,2024,7,False
1,82,72,18,1202,75.0,22.25,768,156.51,0,3,3,70,90,Asif Ali,IU/PZ,2016,2024,8,False
2,75,74,2,1972,107.0,27.38,1440,136.94,3,12,8,213,89,Kamran Akmal,PZ,2016,2022,6,True


### 5.4 Clean: `most_wickets`

In [55]:
df = most_wickets.copy()

df['player_name']  = df['Player'].apply(clean_player_name)
df['teams_played'] = df['Player'].apply(extract_teams)
df['span_start']   = df['Span'].apply(parse_span_start)
df['span_end']     = df['Span'].apply(parse_span_end)
df['career_yrs']   = df['span_end'] - df['span_start']

# BBI: '4/17' → split into bbi_wkts and bbi_runs
df['bbi_wkts'] = df['BBI'].apply(lambda x: int(str(x).split('/')[0]) if '/' in str(x) else np.nan)
df['bbi_runs'] = df['BBI'].apply(lambda x: int(str(x).split('/')[1]) if '/' in str(x) else np.nan)

# 4- and 5-wicket hauls, and maidens: '-' → 0
for col in ['4', '5', 'Mdns']:
    df[col] = df[col].apply(dash_to_zero).astype(float).astype('Int64')

df = df.drop(columns=['Player', 'Span', 'BBI'])
df = df.rename(columns={
    'Mat': 'matches', 'Inns': 'innings', 'Balls': 'balls',
    'Overs': 'overs', 'Mdns': 'maidens', 'Runs': 'runs_conceded',
    'Wkts': 'wickets', 'Ave': 'bowling_avg', 'Econ': 'economy',
    'SR': 'bowling_sr', '4': 'four_wicket_hauls', '5': 'five_wicket_hauls',
})

most_wickets_clean = df.copy()
print("most_wickets cleaned | shape:", df.shape)
df.head(3)


most_wickets cleaned | shape: (100, 19)


,matches,innings,balls,overs,maidens,runs_conceded,wickets,bowling_avg,economy,bowling_sr,four_wicket_hauls,five_wicket_hauls,player_name,teams_played,span_start,span_end,career_yrs,bbi_wkts,bbi_runs
0,88,87,1973,328.5,3,2563,113,22.68,7.79,17.46,1,0,Wahab Riaz,PZ,2016,2023,7,4,17
1,82,81,1852,308.4,2,2480,108,22.96,8.03,17.14,4,0,Hasan Ali,IU/KK/PZ,2016,2024,8,4,15
2,71,71,1623,270.3,4,2166,103,21.02,8.00,15.75,3,2,Shaheen Shah Afridi,LQ,2018,2024,6,5,4


### 5.5 Clean: `most_catches`

In [56]:
df = most_catches.copy()

df['player_name']  = df['Player'].apply(clean_player_name)
df['teams_played'] = df['Player'].apply(extract_teams)
df['span_start']   = df['Span'].apply(parse_span_start)
df['span_end']     = df['Span'].apply(parse_span_end)
df['career_yrs']   = df['span_end'] - df['span_start']

df = df.drop(columns=['Player', 'Span'])
df = df.rename(columns={
    'Mat': 'matches', 'Inns': 'innings',
    'Ct': 'catches', 'Max': 'max_in_match', 'Ct/Inn': 'catches_per_inn',
})

most_catches_clean = df.copy()
print("most_catches cleaned | shape:", df.shape)
df.head(3)


most_catches cleaned | shape: (100, 10)


,matches,innings,catches,max_in_match,catches_per_inn,player_name,teams_played,span_start,span_end,career_yrs
0,90,89,47,4,0.528,Babar Azam,IU/KK/PZ,2016,2024,8
1,53,52,42,4,0.807,KA Pollard,KK/MS/PZ,2017,2024,7
2,84,84,39,3,0.464,Mohammad Nawaz,KK/QG,2016,2024,8


### 5.6 Clean: `most_dismissals` (Wicket-Keeper)

In [57]:
df = most_dismissals.copy()

df['player_name']  = df['Player'].apply(clean_player_name)
df['teams_played'] = df['Player'].apply(extract_teams)
df['span_start']   = df['Span'].apply(parse_span_start)
df['span_end']     = df['Span'].apply(parse_span_end)
df['career_yrs']   = df['span_end'] - df['span_start']

# Dis, Ct, St, Dis/Inn: '-' means 0 (player with no recorded dismissals yet)
for col in ['Dis', 'Ct', 'St', 'Dis/Inn']:
    df[col] = df[col].apply(dash_to_zero)

# 'Max Dis Inns' like '4 (3ct 1st)' — extract numeric total only.
# Players with zero dismissals show '-' here too.
def parse_max_dis(x):
    x = str(x).strip()
    if x == '-' or x == 'nan':
        return 0
    return int(x.split(' ')[0])

df['max_dis_total'] = df['Max Dis Inns'].apply(parse_max_dis)

df = df.drop(columns=['Player', 'Span', 'Max Dis Inns'])
df = df.rename(columns={
    'Mat': 'matches', 'Inns': 'innings', 'Dis': 'total_dismissals',
    'Ct': 'caught', 'St': 'stumped', 'Dis/Inn': 'dismissals_per_inn',
})

most_dismissals_clean = df.copy()
print("most_dismissals cleaned | shape:", df.shape)
df.head(3)


most_dismissals cleaned | shape: (44, 12)


,matches,innings,total_dismissals,caught,stumped,dismissals_per_inn,player_name,teams_played,span_start,span_end,career_yrs,max_dis_total
0,83,80,83.0,68.0,15.0,1.037,Mohammad Rizwan,KK/LQ/MS,2016,2024,8,4
1,75,72,62.0,53.0,9.0,0.861,Kamran Akmal,PZ,2016,2022,6,4
2,86,85,56.0,43.0,13.0,0.658,Sarfaraz Ahmed,QG,2016,2024,8,3


### 5.7 Clean: `highest_score` & `most_sixes_innings` (Match-Level)

In [58]:
def clean_match_batting(df, source_name):
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]
    
    # Runs: '145*' → 145 + flag
    df['is_not_out'] = df['Runs'].apply(is_not_out)
    df['Runs']       = df['Runs'].apply(clean_hs)

    # Balls, Mins, 4s, 6s: '-' → NaN (boundary counts occasionally missing in source)
    for col in ['Balls', 'Mins', '4s', '6s']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].replace('-', np.nan), errors='coerce')

    # SR: some may have '-'
    df['SR'] = pd.to_numeric(df['SR'].replace('-', np.nan), errors='coerce')
    
    # Date
    df['match_date']  = df['Match Date'].apply(parse_date)
    df['match_year']  = df['match_date'].dt.year
    df['match_month'] = df['match_date'].dt.month
    
    # Standardise Opposition (remove leading 'v ')
    df['Opposition'] = df['Opposition'].str.replace(r'^v\s+', '', regex=True).str.strip()

    df = df.drop(columns=['Match Date'])
    df.columns = [c.lower().replace(' ', '_') for c in df.columns]
    
    print(f"  {source_name} cleaned | shape: {df.shape}")
    return df

highest_score_clean      = clean_match_batting(highest_score, 'highest_score')
most_sixes_innings_clean = clean_match_batting(most_sixes_innings, 'most_sixes_innings')

highest_score_clean.head(3)


  highest_score cleaned | shape: (100, 14)
  most_sixes_innings cleaned | shape: (100, 14)


,player,runs,mins,balls,4s,6s,sr,team,opposition,ground,is_not_out,match_date,match_year,match_month
0,JJ Roy,145.0,90.0,63,20,5.0,230.15,Gladiators,Zalmi,Rawalpindi,True,2023-03-08,2023,3
1,CA Ingram,127.0,NaN,59,12,8.0,215.25,Kings,Gladiators,Sharjah,True,2019-02-24,2019,2
2,RR Rossouw,121.0,87.0,51,12,8.0,237.25,Sultans,Zalmi,Rawalpindi,False,2023-03-10,2023,3


### 5.8 Clean: `best_bowling` (Match-Level)

In [59]:
df = best_bowling.copy()
df.columns = [c.strip() for c in df.columns]

# Overs: ensure numeric
df['Overs'] = pd.to_numeric(df['Overs'], errors='coerce')

# Mdns: '-' → 0
df['Mdns'] = df['Mdns'].replace('-', 0).apply(pd.to_numeric, errors='coerce')

# Inns: bowling innings number (1 or 2)
df['Inns'] = pd.to_numeric(df['Inns'], errors='coerce')

# Opposition: remove 'v '
df['Opposition'] = df['Opposition'].str.replace(r'^v\s+', '', regex=True).str.strip()

# Date
df['match_date']  = df['Match Date'].apply(parse_date)
df['match_year']  = df['match_date'].dt.year
df = df.drop(columns=['Match Date'])

df.columns = [c.lower().replace(' ', '_') for c in df.columns]

best_bowling_clean = df.copy()
print("best_bowling cleaned | shape:", df.shape)
df.head(3)


best_bowling cleaned | shape: (100, 12)


,player,overs,mdns,runs,wkts,econ,team,inns,opposition,ground,match_date,match_year
0,RS Bopara,4.0,0,16,6,4.00,Kings,2,Qalandars,Sharjah,2016-02-12,2016
1,Faheem Ashraf,4.0,0,19,6,4.75,United,2,Qalandars,Karachi,2019-03-09,2019
2,Umar Gul,4.0,0,24,6,6.00,Sultans,2,Gladiators,Dubai (DICS),2018-03-07,2018


### 5.9 Clean: `highest_totals` & `lowest_totals`

In [60]:
def clean_team_totals(df, source_name):
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]
    
    # Score: '262/3' → runs + wickets
    df['runs']    = df['Score'].apply(parse_score_runs)
    df['wickets'] = df['Score'].apply(parse_score_wickets)
    df = df.drop(columns=['Score'])

    # Overs: numeric
    df['Overs'] = pd.to_numeric(df['Overs'], errors='coerce')

    # Opposition: remove 'v '
    df['Opposition'] = df['Opposition'].str.replace(r'^v\s+', '', regex=True).str.strip()

    # Date
    df['match_date']  = df['Match Date'].apply(parse_date)
    df['match_year']  = df['match_date'].dt.year
    df = df.drop(columns=['Match Date'])

    # Result: 'won' / 'lost' → bool
    df['won'] = df['Result'].str.lower().str.strip() == 'won'
    df = df.drop(columns=['Result'])

    df.columns = [c.lower().replace(' ', '_') for c in df.columns]
    print(f" {source_name} cleaned | shape: {df.shape}")
    return df

highest_totals_clean = clean_team_totals(highest_totals, 'highest_totals')
lowest_totals_clean  = clean_team_totals(lowest_totals,  'lowest_totals')
highest_totals_clean.head(3)


 highest_totals cleaned | shape: (100, 11)
 lowest_totals cleaned | shape: (200, 11)


,team,overs,rr,inns,opposition,ground,runs,wickets,match_date,match_year,won
0,Sultans,20.0,13.10,1,Gladiators,Rawalpindi,262,3,2023-03-11,2023,True
1,Gladiators,20.0,12.65,2,Sultans,Rawalpindi,253,8,2023-03-11,2023,False
2,United,20.0,12.35,1,Zalmi,Abu Dhabi,247,2,2021-06-17,2021,True


### 5.10 Clean: `result_summary`

In [61]:
df = result_summary.copy()
df.columns = [c.strip() for c in df.columns]

# Rename confusing column '%' (win %) 
# Headers: Team,Span,Mat,Won,Lost,Draw,Tied,Tie+W,Tie+L,NR,W/L,%W,%L,%D,%
df = df.rename(columns={
    'Mat': 'matches', 'Won': 'won', 'Lost': 'lost', 'Draw': 'draws',
    'Tied': 'tied', 'Tie+W': 'tie_wins', 'Tie+L': 'tie_losses',
    'NR': 'no_result', 'W/L': 'win_loss_ratio',
    '%W': 'win_pct', '%L': 'loss_pct', '%D': 'draw_pct', '%': 'overall_pct',
})

df['span_start'] = df['Span'].apply(parse_span_start)
df['span_end']   = df['Span'].apply(parse_span_end)
df = df.drop(columns=['Span'])

# win_loss_ratio: convert string to float
df['win_loss_ratio'] = pd.to_numeric(df['win_loss_ratio'], errors='coerce')

df.columns = [c.lower().replace(' ', '_') for c in df.columns]

result_summary_clean = df.copy()
print("  result_summary cleaned | shape:", df.shape)
df


  result_summary cleaned | shape: (6, 16)


,team,matches,won,lost,draws,tied,tie_wins,tie_losses,no_result,win_loss_ratio,win_pct,loss_pct,draw_pct,overall_pct,span_start,span_end
0,Islamabad United,100,55,44,0,0,1,0,0,1.250,55.00,44.00,0.0,55.50,2016,2024
1,Karachi Kings,95,36,55,0,0,1,1,2,0.654,37.89,57.89,0.0,39.78,2016,2024
2,Lahore Qalandars,94,40,51,0,0,1,2,0,0.784,42.55,54.25,0.0,44.14,2016,2024
3,Multan Sultans,79,45,31,0,0,0,1,2,1.451,56.96,39.24,0.0,59.09,2018,2024
4,Peshawar Zalmi,104,55,47,0,0,1,0,1,1.170,52.88,45.19,0.0,53.88,2016,2024
5,Quetta Gladiators,92,44,47,0,0,0,0,1,0.936,47.82,51.08,0.0,48.35,2016,2024


### 5.11 Clean: `match_wins` (Cumulative Timeline)

In [62]:
df = match_wins.copy()

# The first column is unnamed (match number label)
first_col = df.columns[0]
df = df.rename(columns={first_col: 'match_number'})

# Extract the numeric match number
df['match_number'] = df['match_number'].str.extract(r'(\d+)').astype(int)

# Rename team columns to full names
team_rename = {
    'Quetta'   : 'Quetta Gladiators',
    'Karachi'  : 'Karachi Kings',
    'Peshawar' : 'Peshawar Zalmi',
    'Islamabad': 'Islamabad United',
    'Lahore'   : 'Lahore Qalandars',
    'Multan'   : 'Multan Sultans',
}
df = df.rename(columns=team_rename)

# All win columns should be integer
team_cols = [c for c in df.columns if c != 'match_number']
for col in team_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

match_wins_clean = df.copy()
print(" match_wins cleaned | shape:", df.shape)
df.head()


 match_wins cleaned | shape: (278, 7)


,match_number,Quetta Gladiators,Karachi Kings,Peshawar Zalmi,Islamabad United,Lahore Qalandars,Multan Sultans
0,1,1,0,0,0,0,0
1,2,1,1,0,0,0,0
2,3,1,1,1,0,0,0
3,4,2,1,1,0,0,0
4,5,2,1,2,0,0,0


## 6. Data Consistency Checks

In [63]:
print("=" * 60)
print("  CONSISTENCY CHECKS")
print("=" * 60)

# 1. Check: runs vs innings × average (approximate)
df = most_runs_clean.copy()
df['avg_check'] = (df['runs'] / (df['innings'] - df['not_outs'])).round(2)
mismatch = df[abs(df['avg_check'] - df['average']) > 1][['player_name', 'average', 'avg_check']]
print(f"\n1. most_runs — average vs computed avg: {len(mismatch)} mismatches (small rounding diffs OK)")

# 2. Match date range in highest_score
print(f"\n2. highest_score — date range: "
      f"{highest_score_clean['match_date'].min().date()} to "
      f"{highest_score_clean['match_date'].max().date()}")

# 3. Teams in result_summary
print(f"\n3. result_summary — teams: {result_summary_clean['team'].tolist()}")

# 4. Score ranges
print(f"\n4. highest_totals — runs range: {highest_totals_clean['runs'].min()}–{highest_totals_clean['runs'].max()}")
print(f"   lowest_totals  — runs range: {lowest_totals_clean['runs'].min()}–{lowest_totals_clean['runs'].max()}")

# 5. Wickets in best_bowling
print(f"\n5. best_bowling — wickets range: {best_bowling_clean['wkts'].min()}–{best_bowling_clean['wkts'].max()}")

print("\n All checks passed")


  CONSISTENCY CHECKS

1. most_runs — average vs computed avg: 0 mismatches (small rounding diffs OK)

2. highest_score — date range: 2016-02-04 to 2024-03-10

3. result_summary — teams: ['Islamabad United', 'Karachi Kings', 'Lahore Qalandars', 'Multan Sultans', 'Peshawar Zalmi', 'Quetta Gladiators']

4. highest_totals — runs range: 190–262
   lowest_totals  — runs range: 59–164

5. best_bowling — wickets range: 3–6

 All checks passed


In [64]:
# Standardize team & opposition names to full franchise names (raw data only gives the last word)
team_map = {
    'United'    : 'Islamabad United',
    'Kings'     : 'Karachi Kings',
    'Qalandars' : 'Lahore Qalandars',
    'Sultans'   : 'Multan Sultans',
    'Zalmi'     : 'Peshawar Zalmi',
    'Gladiators': 'Quetta Gladiators',
}

for df in [highest_score_clean, most_sixes_innings_clean, best_bowling_clean,
           highest_totals_clean, lowest_totals_clean]:
    df['team'] = df['team'].map(team_map).fillna(df['team'])
    if 'opposition' in df.columns:
        df['opposition'] = df['opposition'].map(team_map).fillna(df['opposition'])

print("Team/opposition names standardized to full franchise names")

Team/opposition names standardized to full franchise names


## 7. Save Cleaned Datasets

In [65]:
import os
os.makedirs(PROC, exist_ok=True)

saves = {
    'most_runs_clean':            most_runs_clean,
    'most_sixes_history_clean':   most_sixes_history_clean,
    'most_wickets_clean':         most_wickets_clean,
    'most_catches_clean':         most_catches_clean,
    'most_dismissals_clean':      most_dismissals_clean,
    'highest_score_clean':        highest_score_clean,
    'most_sixes_innings_clean':   most_sixes_innings_clean,
    'best_bowling_clean':         best_bowling_clean,
    'highest_totals_clean':       highest_totals_clean,
    'lowest_totals_clean':        lowest_totals_clean,
    'result_summary_clean':       result_summary_clean,
    'match_wins_clean':           match_wins_clean,
}

for fname, df in saves.items():
    df.to_csv(PROC + fname + '.csv', index=False)
    print(f"  {fname}.csv  → {df.shape}")

print("\n All cleaned datasets saved to ../data/preprocessed/")


  most_runs_clean.csv  → (150, 19)
  most_sixes_history_clean.csv  → (100, 19)
  most_wickets_clean.csv  → (100, 19)
  most_catches_clean.csv  → (100, 10)
  most_dismissals_clean.csv  → (44, 12)
  highest_score_clean.csv  → (100, 14)
  most_sixes_innings_clean.csv  → (100, 14)
  best_bowling_clean.csv  → (100, 12)
  highest_totals_clean.csv  → (100, 11)
  lowest_totals_clean.csv  → (200, 11)
  result_summary_clean.csv  → (6, 16)
  match_wins_clean.csv  → (278, 7)

 All cleaned datasets saved to ../data/preprocessed/


## 8. Export Cleaned Data to Excel (Multi-Sheet Workbook)

In addition to individual CSVs, save all 12 cleaned tables into a **single Excel workbook** — one sheet per dataset. This makes the cleaned data easy to share with non-technical stakeholders and to open directly in Excel.

In [66]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

excel_path = PROC + 'psl_cleaned_data.xlsx'

HEADER_FILL = PatternFill('solid', start_color='1F4E78', end_color='1F4E78')
HEADER_FONT = Font(name='Arial', bold=True, color='FFFFFF', size=11)
BODY_FONT   = Font(name='Arial', size=10)

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    for fname, df in saves.items():
        sheet_name = fname.replace('_clean', '')[:31]  # Excel sheet name limit
        df.to_excel(writer, sheet_name=sheet_name, index=False)

# Re-open to apply formatting
wb = openpyxl.load_workbook(excel_path)
for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]
    # Style header row
    for cell in ws[1]:
        cell.fill = HEADER_FILL
        cell.font = HEADER_FONT
        cell.alignment = Alignment(horizontal='center', vertical='center')
    # Style body + autosize columns
    for col_idx, col_cells in enumerate(ws.columns, start=1):
        max_len = max((len(str(c.value)) if c.value is not None else 0) for c in col_cells)
        ws.column_dimensions[get_column_letter(col_idx)].width = min(max_len + 3, 35)
        for cell in col_cells[1:]:
            cell.font = BODY_FONT
    ws.freeze_panes = 'A2'

wb.save(excel_path)
print(f"  Excel workbook saved: {excel_path}")
print(f"   Sheets: {wb.sheetnames}")


  Excel workbook saved: ../data/preprocessed/psl_cleaned_data.xlsx
   Sheets: ['most_runs', 'most_sixes_history', 'most_wickets', 'most_catches', 'most_dismissals', 'highest_score', 'most_sixes_innings', 'best_bowling', 'highest_totals', 'lowest_totals', 'result_summary', 'match_wins']


## 9. Summary of Cleaning Performed

| Task | Details |
|------|--------|
| Column standardisation | All column names lowercased, spaces → underscores |
| Player name parsing | Extracted clean name and team codes into separate columns |
| Span parsing | `'2016-2024'` → `span_start=2016`, `span_end=2024`, `career_yrs=8` |
| Not-out flag | `'115*'` → `high_score=115`, `hs_not_out=True` |
| Dash handling | `'-'` in century/50 counts → `0`; in Balls/Mins → `NaN` |
| Score parsing | `'262/3'` → `runs=262`, `wickets=3`; no `/` → `wickets=10` (all out) |
| BBI parsing | `'4/17'` → `bbi_wkts=4`, `bbi_runs=17` |
| Date conversion | `'08 Mar 2023'` → `pd.Timestamp`, plus `match_year` and `match_month` |
| Opposition cleaning | Removed leading `'v '` prefix |
| Timeline reshape | Named match-number column, expanded team names to full franchise names |
| Max Dis Inns | Extracted numeric total from complex string `'4 (3ct 1st)'` |